In [1]:
from sqlalchemy import (
    Boolean,
    Column,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    text,
    update,
)

from egomimic.utils.aws.aws_sql import (
    TableRow,
    create_default_engine,
    episode_table_to_df,
)

In [3]:
engine = create_default_engine()

Tables in schema 'app': ['episodes']


In [5]:
df = episode_table_to_df(engine)



In [6]:
import pandas as pd

# 1）每个 embodiment 有多少行
counts = df.groupby("embodiment", dropna=False).size()
# 或：counts = df["embodiment"].value_counts(dropna=False)

# 2）每个 embodiment 下有多少个不同的 task
unique_tasks = df.groupby("embodiment", dropna=False)["task"].nunique()

print("每条 embodiment 的记录数:")
print(counts)
print("\n每条 embodiment 的 unique task 数:")
print(unique_tasks)

每条 embodiment 的记录数:
embodiment
                      1
aria               2435
eva                3394
mecka             41617
scale             24813
scale_bimanual        1
NaN                   3
dtype: int64

每条 embodiment 的 unique task 数:
embodiment
                     1
aria                21
eva                 18
mecka             4859
scale              120
scale_bimanual       1
NaN                  1
Name: task, dtype: int64


In [11]:
import pandas as pd

# 只统计明确为 True 的行
n = (df["is_deleted"] == True).sum()
# 或
n = df["is_deleted"].eq(True).sum()
n

np.int64(7905)

In [18]:
# 在df["is_deleted"]的数据中，每类df["embodiment"]的df['num_frames']的加和是多少
import pandas as pd

mask = df["is_deleted"].eq(False)
sums = (
    df.loc[mask]
    .groupby("embodiment", dropna=False)["num_frames"]
    .sum()
    .rename("valid_frames_sum")
    .reset_index()
)

print(sums)

  embodiment  valid_frames_sum
0       aria        11266125.0
1        eva         3563037.0
2      mecka        89466016.0
3      scale        23389757.0


In [15]:
import pandas as pd

# 只统计已删除
deleted = df.loc[df["is_deleted"].eq(True)]

counts = deleted.groupby("embodiment", dropna=False).size().rename("n_deleted").reset_index()

# 若也想看未删除为 0 的任务，可从全体 task 左连接补全
all_tasks = df[["embodiment"]].drop_duplicates()
counts_full = all_tasks.merge(counts, on="embodiment", how="left").fillna({"n_deleted": 0}).astype({"n_deleted": int})

print(counts)        # 仅出现过删除记录的任务
print(counts_full)   # 所有出现过的 task，无删除则为 0

       embodiment  n_deleted
0                          1
1            aria          1
2             eva        176
3           scale       7723
4  scale_bimanual          1
5             NaN          3
       embodiment  n_deleted
0           mecka          0
1           scale       7723
2             eva        176
3            aria          1
4             NaN          3
5                          1
6  scale_bimanual          1


In [9]:
import pandas as pd

# 只看未删除的行
alive = df[df["is_deleted"].eq(False)]

# 1) 每个 embodiment 有多少个不同的 task
by_embodiment = (
    alive.groupby("embodiment", dropna=False)["task"]
    .nunique()
    .rename("unique_task_count")
    .reset_index()
)

# 2) 每个 task 有多少条数据
by_task = (
    alive.groupby("task", dropna=False).size()
    .rename("row_count")
    .reset_index()
)

# 保存为 CSV（可自行改路径）
# by_embodiment.to_csv("embodiment_unique_tasks.csv", index=False)
# by_task.to_csv("task_row_counts.csv", index=False)

print(by_embodiment)
print(by_task)

  embodiment  unique_task_count
0       aria                 20
1        eva                 15
2      mecka               4859
3      scale                 71
                               task  row_count
0                                            6
1                     accessorizing          2
2            accessorizing_bouquets          5
3               accessorizing_gifts          2
4     adding_and_mixing_ingredients          2
...                             ...        ...
4950          writing_on_cake_board          1
4951               writing_on_forms          1
4952            writing_on_stickers          1
4953                zipping_pillows          2
4954                  zipping_pouch          5

[4955 rows x 2 columns]


In [ ]:
# 在df["is_deleted"]的数据中，每类df["embodiment"]的df['num_frames']的加和是多少
import pandas as pd

mask = df["is_deleted"].eq(True)
sums = (
    df.loc[mask]
    .groupby("embodiment", dropna=False)["num_frames"]
    .sum()
    .rename("num_frames_sum_deleted")
    .reset_index()
)

print(sums)

In [8]:
import pandas as pd

mask = df["is_deleted"].eq(False)  # 只统计未删除
d = df.loc[mask].copy()

# 1）每个 embodiment：行数 + 不同 task 个数
by_emb = (
    d.groupby("embodiment", dropna=False)
    .agg(
        n_rows=("task", "size"),
        n_unique_tasks=("task", "nunique"),
    )
    .reset_index()
)
by_emb.to_csv("embodiment_stats.csv", index=False)

# 2）每个 (embodiment, task)：行数
by_emb_task = (
    d.groupby(["embodiment", "task"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
)
by_emb_task.to_csv("embodiment_task_row_counts.csv", index=False)

print(by_emb.head())
print(by_emb_task.head())

  embodiment  n_rows  n_unique_tasks
0       aria    2434              20
1        eva    3218              15
2      mecka   41617            4859
3      scale   17090              71
  embodiment                     task  n_rows
0       aria            bag_groceries     144
1       aria  bag_groceries_in_domain      21
2       aria    bag_groceries_success      24
3       aria            cup_on_saucer     554
4       aria  cup_on_saucer_in_domain      20


In [21]:
a = 11266125/2434/30
b = 3563037/3218/30
c = 89466016/41617/30
d = 23389757/17090/30
print(a, b, c, d)

154.28820870994247 36.90736482287135 71.65822940945607 45.620747025551005


In [12]:
df

,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,...,robot_name,is_eval,eval_score,eval_success,processing_error,zarr_processed_path,zarr_processing_error,zarr_mp4_path,license,segments
0,696bc6351cafc9f413b5b7e8,695d098f83a9fdf2d84d9a47,mecka,1960.0,packaging_cloths,"In the final part of the video, the person con...",stockroom,"[""plaid_cloths"", ""plastic_bags"", ""cardboard_bo...",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696bc635...,,s3://rldb/processed_v3/mecka/freeform/696bc635...,NaN,None
1,696b2152d1aa1f60b4db5ef1,694a6ead5c0479a0fb867619,mecka,2700.0,cleaning_engine_parts,"A mechanic continues to clean a used, three-cy...",service_bay,"[""engine_gasket"", ""workbench"", ""cloth""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696b2152...,,s3://rldb/processed_v3/mecka/freeform/696b2152...,NaN,None
2,696bcb732a9a68766920de81,696452cf83a9fdf2d86552c5,mecka,2252.0,folding_paper_bags,"A person begins folding a new paper bag, caref...",packaging_area,"[""paper_sheet"", ""glue"", ""plastic_container"", ""...",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696bcb73...,,s3://rldb/processed_v3/mecka/freeform/696bcb73...,NaN,None
3,69b4944484179943aec080c6,6948bbeb5c0479a0fb82ec42,mecka,2278.0,planting_cuttings,Planting plant cuttings into a tray filled wit...,nursery,"[""soil"", ""plant_cuttings"", ""tray"", ""pots""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/69b49444...,,s3://rldb/processed_v3/mecka/freeform/69b49444...,CC BY-SA 4.0,"[{'label': 'adjust soil in container', 'end_se..."
4,693cd014bec8abdc94f9a365,693369d4c195d3a38a24af5c,mecka,2650.0,labeling_eggs,The person uses a pair of scissors to cut labe...,packing_shed,"[""labels"", ""egg_cartons"", ""eggs""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/693cd014...,,s3://rldb/processed_v3/mecka/freeform/693cd014...,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72241,696db125ddefc568d7372680,695b04fc83a9fdf2d8485831,mecka,2698.0,packaging_items,"A person is sitting and packaging small, black...",material_storage,"[""air_stones"", ""plastic_bags"", ""cardboard_box""...",s3://rldb/mecka/freeform/696db125ddefc568d7372680,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696db125...,,s3://rldb/processed_v3/mecka/freeform/696db125...,CC BY-SA 4.0,"[{'label': 'pick up plastic bag, pick up grey ..."
72242,696b9a50a04610abd4bb2ed0,694901865c0479a0fb833c40,mecka,2848.0,decorating_party_horns,A person attaches a pre-made blue fringed pape...,craft_section,"[""party_horn"", ""blue_fringed_paper"", ""white_fr...",s3://rldb/mecka/freeform/696b9a50a04610abd4bb2ed0,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696b9a50...,,s3://rldb/processed_v3/mecka/freeform/696b9a50...,CC BY-SA 4.0,"[{'label': 'wrap fringe, adjust cone', 'end_se..."
72243,696bac1d7e6f94bf3414388d,6951c933c0d9448524bb1d46,mecka,3598.0,making_kites,A person continues to assemble a kite frame. T...,craft_section,"[""kite_frame"", ""bamboo_sticks"", ""string""]",s3://rldb/mecka/freeform/696bac1d7e6f94bf3414388d,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696bac1d...,,s3://rldb/processed_v3/mecka/freeform/696bac1d...,CC BY-SA 4.0,[{'label': 'tie two wooden sticks to form a cr...
72244,696babf5b17075c2c32fa74b,694c877362d08e8cd95f46fd,mecka,3598.0,cleaning_shoes,A person is using a brush and soapy water to s...,repair_bench,"[""shoe"", ""soapy_water""]",s3://rldb/mecka/freeform/696babf5b17075c2c32fa74b,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696babf5...,,s3://rldb/processed_v3/mecka/freeform/696babf5...,CC BY-SA 4.0,"[{'label': 'hold shoe, scrub shoe with brush',..."


In [36]:
df.columns.tolist()

['episode_hash',
 'operator',
 'lab',
 'num_frames',
 'task',
 'task_description',
 'scene',
 'objects',
 'processed_path',
 'mp4_path',
 'is_deleted',
 'embodiment',
 'robot_name',
 'is_eval',
 'eval_score',
 'eval_success',
 'processing_error',
 'zarr_processed_path',
 'zarr_processing_error',
 'zarr_mp4_path',
 'license',
 'segments']

In [37]:
df['eval_success'].unique()

array([True, None, False], dtype=object)

In [40]:
df.to_csv("output.csv", index=False, encoding="utf-8")

In [39]:
df['processing_error']

0         
1         
2         
3         
4         
        ..
72241     
72242     
72243     
72244     
72245     
Name: processing_error, Length: 72246, dtype: str

In [9]:
df["task"].unique()

<ArrowStringArray>
[             'packaging_cloths',         'cleaning_engine_parts',
            'folding_paper_bags',             'planting_cuttings',
                 'labeling_eggs',        'flagship_sort_utensils',
               'folding_clothes',                'cleaning_shoes',
   'filling_pot_with_rice_husks',                    'pick_place',
 ...
                   'gluing_toys',              'brushing_pistons',
               'knitting_fabric', 'crafting_miniature_structures',
                 'sanding_gears',       'opening_perfume_bottles',
        'mixing_scented_liquids',                 'cutting_pipes',
            'untangling_threads',            'sleeving_banknotes']
Length: 5009, dtype: str

In [ ]:
# Filter to certain lab
df[df["lab"] == "rl2"]

In [28]:
import json
from pathlib import Path

out = Path("tasks_by_num_frames_desc.json")
ordered = df.sort_values("num_frames", ascending=False)["task"].tolist()

out.write_text(json.dumps(ordered, ensure_ascii=False, indent=2), encoding="utf-8")

1843187

In [34]:
import json
from pathlib import Path

out = Path("tasks_and_num_frames_desc.json")
records = (
    df.sort_values("num_frames", ascending=False)[["task", "num_frames"]]
    .to_dict(orient="records")
)

out.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

4998566

In [35]:
import json
from pathlib import Path

out = Path("tasks_frames_sum_desc.json")  # 改成你的路径

agg = (
    df.groupby("task", as_index=False)["num_frames"]
    .sum()
    .rename(columns={"num_frames": "frames"})
    .sort_values("frames", ascending=False)
)

out.write_text(
    json.dumps(agg.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

324093

In [26]:
import json

df["task"].tolist()
with open("tasks.json", "w", encoding="utf-8") as f:
    json.dump(df["task"].tolist(), f, ensure_ascii=False, indent=2)

In [25]:
import pandas as pd

# 每个 scene 的 num_frames 之和
per_scene = df.groupby("task", as_index=False)["num_frames"].sum()

# 只看这一列的统计量（pandas 自带）
s = per_scene["num_frames"]

stats = {
    "count": s.count(),           # scene 个数
    "sum": s.sum(),               # 全体总帧数（= 原表 num_frames 总和）
    "max": s.max(),
    "min": s.min(),
    "mean": s.mean(),
    "median": s.median(),
    "std": s.std(),               # 样本标准差（ddof=1）
    "var": s.var(),               # 样本方差（ddof=1）
}

# 或一行汇总
summary = s.describe()  # count, mean, std, min, 25%, 50%, 75%, max
per_scene.sort_values("num_frames", ascending=True).head(50)

,task,num_frames
2627,move_caprisun,-81.0
5006,zarr_test,-5.0
610,bimanual_test,-1.0
1,TESTING A VIDEO UPLOADa,0.0
4452,test,20.0
2049,freeform_take_a_picture_with_camera,378.0
2056,freeform_tong_gripper_food_transfer,399.0
4453,test_data,405.0
1991,freeform_insert_usb_cable_into_port,428.0
2021,freeform_plug_in_a_charger_to_a_device,461.0


In [21]:
df[df["scene"] == "unknown"]

,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,...,robot_name,is_eval,eval_score,eval_success,processing_error,zarr_processed_path,zarr_processing_error,zarr_mp4_path,license,segments
5,2026-04-30-09-16-32-579211,scale,scale,873.0,flagship_sort_utensils,Sort utensils,unknown,,,,...,scale_right_arm,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-04-30-09-16-...,,s3://rldb/processed_v3/scale/2026-04-30-09-16-...,NaN,None
16,2026-04-30-09-17-09-518391,scale,scale,511.0,freeform_placing_utensils_into_a_drawer,Placing utensils into a drawer,unknown,,,,...,scale_left_arm,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-04-30-09-17-...,,s3://rldb/processed_v3/scale/2026-04-30-09-17-...,NaN,None
36,2026-04-30-09-16-42-987257,scale,scale,1287.0,flagship_sort_utensils,Sort utensils,unknown,,,,...,scale_right_arm,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-04-30-09-16-...,,s3://rldb/processed_v3/scale/2026-04-30-09-16-...,NaN,None
38,2026-03-18-22-53-54-620571,scale,scale,6147.0,freeform_unpack_groceries,Unpack grocery bag,unknown,,,,...,scale_bimanual,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-03-18-22-53-...,,s3://rldb/processed_v3/scale/2026-03-18-22-53-...,CC BY-SA 4.0,None
71,2026-04-30-09-22-56-839549,scale,scale,691.0,flagship_sort_utensils,Sort utensils,unknown,,,,...,scale_right_arm,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-04-30-09-22-...,,s3://rldb/processed_v3/scale/2026-04-30-09-22-...,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71367,2026-03-18-09-10-49-616488,scale,scale,12671.0,freeform_object_in_container,Put Object in Container,unknown,,,,...,scale_bimanual,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-03-18-09-10-...,,s3://rldb/processed_v3/scale/2026-03-18-09-10-...,CC BY-SA 4.0,None
71370,2026-03-18-17-37-26-393860,scale,scale,843.0,flagship_bag_groceries,Bagging groceries,unknown,,,,...,scale_bimanual,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-03-18-17-37-...,,s3://rldb/processed_v3/scale/2026-03-18-17-37-...,CC BY-SA 4.0,None
71371,2026-03-18-18-50-27-958768,scale,scale,669.0,flagship_fold_clothes,Folding Clothes,unknown,,,,...,scale_bimanual,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-03-18-18-50-...,,s3://rldb/processed_v3/scale/2026-03-18-18-50-...,CC BY-SA 4.0,None
71374,2026-03-18-19-03-26-038397,scale,scale,1106.0,flagship_fold_clothes,Folding Clothes,unknown,,,,...,scale_bimanual,False,-1.0,True,,s3://rldb/processed_v3/scale/2026-03-18-19-03-...,,s3://rldb/processed_v3/scale/2026-03-18-19-03-...,CC BY-SA 4.0,None


In [16]:
df.groupby("scene", as_index=False)["num_frames"].sum()

,scene,num_frames
0,,1000847.0
1,1,3661648.0
2,10,306811.0
3,11,145884.0
4,12,125501.0
...,...,...
399,workshop_area,578123.0
400,workspace,8703.0
401,workstation,8494.0
402,workstation_area,78158.0


In [14]:
df[(df["scene"] == "stockroom")]

,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,...,robot_name,is_eval,eval_score,eval_success,processing_error,zarr_processed_path,zarr_processing_error,zarr_mp4_path,license,segments
0,696bc6351cafc9f413b5b7e8,695d098f83a9fdf2d84d9a47,mecka,1960.0,packaging_cloths,"In the final part of the video, the person con...",stockroom,"[""plaid_cloths"", ""plastic_bags"", ""cardboard_bo...",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696bc635...,,s3://rldb/processed_v3/mecka/freeform/696bc635...,NaN,None
6,696bc11770d628f8b0742337,695d084183a9fdf2d84d955a,mecka,1877.0,folding_clothes,The person spreads and folds a checkered cloth...,stockroom,"[""cloths"", ""plastic_bags"", ""table""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696bc117...,,s3://rldb/processed_v3/mecka/freeform/696bc117...,NaN,None
24,693ce1e6301d41387648a2f0,69358f3ac67c9e4814acb45a,mecka,3002.0,bottling_perfumes,The person continues the repetitive task of tr...,stockroom,"[""metal bottle"", ""glass bottle"", ""tissue""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/693ce1e6...,,s3://rldb/processed_v3/mecka/freeform/693ce1e6...,NaN,None
34,696b2ab290e2545845783ba8,693cbbbbc67c9e4814b2ce46,mecka,3601.0,packaging_clothes,A person is preparing a green and grey striped...,stockroom,"[""shirt"", ""lint_roller"", ""plastic_bag"", ""table""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696b2ab2...,,s3://rldb/processed_v3/mecka/freeform/696b2ab2...,NaN,None
66,696ba04aa6f1d19c018342bf,6938404ec67c9e4814aecc35,mecka,2100.0,bottling_perfumes,The person draws liquid from a silver containe...,stockroom,"[""graduated_cylinder"", ""small_glass_bottle"", ""...",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696ba04a...,,s3://rldb/processed_v3/mecka/freeform/696ba04a...,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72134,69b4e7f2806149738b4e59aa,6905a4e79adc5c8f26f52ca2,mecka,2618.0,ironing_clothes,Ironing a black t-shirt on a wooden surface to...,stockroom,"[""t-shirt"", ""cardboard_boxes""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/69b4e7f2...,,s3://rldb/processed_v3/mecka/freeform/69b4e7f2...,CC BY-SA 4.0,[{'label': 'smoothen black shirt on wooden sur...
72172,69b54f5750ea53288aeda4c5,6905ae9801f5a3ac3bd26f63,mecka,2695.0,ironing_clothes,Ironing a white t-shirt with printed text on a...,stockroom,"[""t-shirt"", ""cardboard_boxes""]",NaN,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/69b54f57...,,s3://rldb/processed_v3/mecka/freeform/69b54f57...,CC BY-SA 4.0,"[{'label': 'iron t-shirt, place iron on table'..."
72225,696e755f2934fcc9b4fff6c6,695cc52983a9fdf2d84cd081,mecka,3598.0,packaging_items,The person pours gold-colored safety pins from...,stockroom,"[""safety_pins"", ""plastic_bag"", ""paper_cup"", ""t...",s3://rldb/mecka/freeform/696e755f2934fcc9b4fff6c6,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696e755f...,,s3://rldb/processed_v3/mecka/freeform/696e755f...,CC BY-SA 4.0,"[{'label': 'pick up plastic bag, open plastic ..."
72227,696e888a11a1ac2628c605e9,693ea3047ebf38771693cc3a,mecka,2698.0,inspecting_phone_screens,The person continues the repetitive task of in...,stockroom,"[""phone_screen_protectors"", ""packaging""]",s3://rldb/mecka/freeform/696e888a11a1ac2628c605e9,NaN,...,mecka_bimanual,False,-1.0,True,,s3://rldb/processed_v3/mecka/freeform/696e888a...,,s3://rldb/processed_v3/mecka/freeform/696e888a...,CC BY-SA 4.0,"[{'label': 'hold smartphone screen', 'end_seco..."


In [13]:
# Flagship Fold Clothes Data
df[(df["task"] == "fold_clothes") & df["lab"].isin(["rl2", "eth", "song", "wang"])]

,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,...,robot_name,is_eval,eval_score,eval_success,processing_error,zarr_processed_path,zarr_processing_error,zarr_mp4_path,license,segments
70,2026-01-03-02-35-09-502000,4OOitkcdBEpTp3V5,rl2,787.0,fold_clothes,fold the t-shirt,1,"{""gray vneck t-shirt""}",s3://rldb/processed_v2/eva/2026-01-03-02-35-09...,rldb:/processed_v2/eva/1767407709502_video.mp4,...,eva_bimanual,False,-1.0,True,,s3://rldb/processed_v3/eva/2026-01-03-02-35-09...,,s3://rldb/processed_v3/eva/2026-01-03-02-35-09...,CC BY-SA 4.0,None
223,2025-10-14-04-15-30-000000,omDamnph0YxUBJSO,rl2,2888.0,fold_clothes,folding clothes,10,"{""short sleeve shirts"",""long sleeve shirts"",sh...",,,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-10-14-04-15-3...,,s3://rldb/processed_v3/aria/2025-10-14-04-15-3...,CC BY-SA 4.0,None
233,2025-10-14-03-46-49-000000,omDamnph0YxUBJSO,rl2,3204.0,fold_clothes,folding clothes,16,"{""short sleeve shirts"",""long sleeve shirts"",sh...",,,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-10-14-03-46-4...,,s3://rldb/processed_v3/aria/2025-10-14-03-46-4...,CC BY-SA 4.0,None
287,2025-12-31-20-34-16-063000,4OOitkcdBEpTp3V5,rl2,1496.0,fold_clothes,fold the t-shirt,1,"{""white medium t-shirt""}",s3://rldb/processed_v2/eva/2025-12-31-20-34-16...,rldb:/processed_v2/eva/1767213256063_video.mp4,...,eva_bimanual,False,-1.0,True,,,,,CC BY-SA 4.0,None
374,2026-04-17-21-20-29-581000,19,rl2,-1.0,fold_clothes,fold the blue shirt,17,"{""blue shirt""}",,,...,eva_bimanual,False,-1.0,True,,,Zero Frames,,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71213,2025-11-11-22-55-28-353000,rl2,rl2,1567.0,fold_clothes,folding clothes,8,"{""short sleeve shirts"",""long sleeve shirts"",sh...",s3://rldb/processed_v2/aria/2025-11-11-22-55-2...,s3://rldb/processed_v2/aria/1762901728353_vide...,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-11-11-22-55-2...,,s3://rldb/processed_v3/aria/2025-11-11-22-55-2...,CC BY-SA 4.0,None
71214,2025-11-11-22-41-52-671000,wang,wang,2605.0,fold_clothes,folding clothes,7,{/coc/cedarp-dxu345-0/datasets/egoverse/old_da...,s3://rldb/processed_v2/aria/2025-11-11-22-41-5...,s3://rldb/processed_v2/aria/1762900912671_vide...,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-11-11-22-41-5...,,s3://rldb/processed_v3/aria/2025-11-11-22-41-5...,CC BY-SA 4.0,None
71215,2025-11-11-23-02-37-422000,song,song,3511.0,fold_clothes,folding clothes,5,"{""short sleeve shirts"",""long sleeve shirts"",sh...",s3://rldb/processed_v2/aria/2025-11-11-23-02-3...,s3://rldb/processed_v2/aria/1762902157422_vide...,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-11-11-23-02-3...,,s3://rldb/processed_v3/aria/2025-11-11-23-02-3...,CC BY-SA 4.0,None
71216,2025-11-11-23-00-10-404000,song,song,4013.0,fold_clothes,folding clothes,4,"{""short sleeve shirts"",""long sleeve shirts"",sh...",s3://rldb/processed_v2/aria/2025-11-11-23-00-1...,s3://rldb/processed_v2/aria/1762902010404_vide...,...,aria_bimanual,False,-1.0,True,,s3://rldb/processed_v3/aria/2025-11-11-23-00-1...,,s3://rldb/processed_v3/aria/2025-11-11-23-00-1...,CC BY-SA 4.0,None
